# Tuned XGBoost + ETR stacking

Run after generating tuned base validation/test prediction CSVs. Change TASK and VARIANT, then run all cells.

In [ ]:
%pip install -q pandas scikit-learn joblib
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/demand-forecasting')
TASK = 'demand'       # demand, price, or supply
VARIANT = 'core'     # core or full
TARGETS = {
    'demand': 'future_demand_kg',
    'price': 'future_weighted_avg_procurement_price_rm_per_kg',
    'supply': 'future_production_kg',
}
TARGET_COLUMN = TARGETS[TASK]
META_FEATURES = ['xgboost_tuned_prediction', 'etr_tuned_prediction']
INPUT_DIR = DRIVE_PROJECT_DIR / 'artifacts' / TASK / VARIANT / 'predictions' / 'tuned_base'
validation_file = INPUT_DIR / 'tuned_base_validation_predictions.csv'
test_file = INPUT_DIR / 'tuned_base_test_predictions.csv'
if not validation_file.exists() or not test_file.exists():
    raise FileNotFoundError(f'Upload tuned base predictions to {INPUT_DIR}')
validation_predictions = pd.read_csv(validation_file)
test_predictions = pd.read_csv(test_file)
required = META_FEATURES + [TARGET_COLUMN]
for name, frame in [('validation', validation_predictions), ('test', test_predictions)]:
    missing = [column for column in required if column not in frame.columns]
    if missing:
        raise ValueError(f'{name} is missing columns: {missing}')
print(f'Task: {TASK}; variant: {VARIANT}')
print('Validation rows:', len(validation_predictions))
print('Test rows:', len(test_predictions))

In [ ]:
X_validation = validation_predictions[META_FEATURES]
y_validation = validation_predictions[TARGET_COLUMN]
X_test = test_predictions[META_FEATURES]
y_test = test_predictions[TARGET_COLUMN]
stacker = Ridge(alpha=1.0)
stacker.fit(X_validation, y_validation)
stacking_prediction = np.maximum(stacker.predict(X_test), 0)
error = y_test.to_numpy() - stacking_prediction
metrics = {
    'mae': float(mean_absolute_error(y_test, stacking_prediction)),
    'rmse': float(np.sqrt(mean_squared_error(y_test, stacking_prediction))),
    'wape': float(np.abs(error).sum() / np.abs(y_test.to_numpy()).sum()),
    'r2': float(r2_score(y_test, stacking_prediction)),
}
print('Test metrics:', metrics)

In [ ]:
OUTPUT_DIR = DRIVE_PROJECT_DIR / 'artifacts' / TASK / VARIANT
(OUTPUT_DIR / 'models').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'reports').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'predictions').mkdir(parents=True, exist_ok=True)
joblib.dump(stacker, OUTPUT_DIR / 'models' / 'tuned_ridge_stacker.joblib')
output = test_predictions.copy()
output['stacking_tuned_prediction'] = stacking_prediction
output['absolute_error'] = np.abs(output[TARGET_COLUMN] - output['stacking_tuned_prediction'])
output.to_csv(OUTPUT_DIR / 'predictions' / 'tuned_stacking_test_predictions.csv', index=False)
metadata = {
    'task': TASK, 'model_variant': VARIANT,
    'model': 'ridge_stacker_on_tuned_xgboost_and_etr',
    'meta_features': META_FEATURES, 'target_column': TARGET_COLUMN,
    'test_metrics': metrics,
}
(OUTPUT_DIR / 'reports' / 'tuned_stacking_metadata.json').write_text(json.dumps(metadata, indent=2))
print('Saved tuned stacking outputs to:', OUTPUT_DIR)